In [36]:
import pandas as pd 
import numpy as np
import requests
from bs4 import BeautifulSoup

In [41]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36"
}
df = pd.DataFrame()
for i in range(1,101):
    webpage = requests.get(f'https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav&page={i}',headers=headers).text
    soup = BeautifulSoup(webpage,'lxml')

    cmp = soup.find_all('div',class_="companyCardWrapper")
    rating = []
    companies = []
    reviews = []
    sector = []
    salary = []
    for company in cmp:
        companies.append(company.find('h2',class_="companyCardWrapper__companyName").text.strip())
        try:
            rating.append(company.find('div',class_="rating_text rating_text--md").text.strip())
        except:
            rating.append(np.nan)

        try:
            reviews.append(company.find('span',class_="companyCardWrapper__companyRatingCount").text.strip()[1:-1])
        except:
            reviews.append(np.nan)
        sector.append(company.find('span',class_="companyCardWrapper__interLinking").text.strip().split("|")[0])
        salary.append(company.find('span',class_="companyCardWrapper__ActionTitle",string="Salaries").find_previous_sibling('span',class_="companyCardWrapper__ActionCount").text.strip())

    data = {
        "Name":companies,
        "sector":sector,
        "rating":rating,
        "reviews":reviews,
        "salaries":salary
    }

    temp_df = pd.DataFrame(data)
    df = pd.concat([df,temp_df],ignore_index=True)

In [47]:
df.rename(columns={
    "Name": "Company",
    "sector": "Sector",
    "rating": "Rating",
    "reviews": "Reviews",
    "salaries": "Salaries"
}, inplace=True)

In [54]:
df.to_excel("Company_Web_Scrapped.xlsx",index=False)